**Engineering foundation branch.** Completed motion helpers and fixed experiment profiles are imported from this repository. The course discussion prompts below remain reference material. Training, scored simulation, and hardware approval are separate statuses.

# CS 123 — Describe a Trick in Words 🐾💬

Type a sentence, get a Pupper motion. [UniMate](https://linzhanmou.com/unimate/)
(Mou et al., SIGGRAPH Asia 2026) is a text-to-motion model that animates *any*
skeleton, including one it has never seen: we describe Pupper's legs to it, and
it moves them the way the sentence says.

The loop:

1. **Describe** a move in a sentence (or a few in a row). UniMate makes several samples.
2. **Pick** the best sample, and **fit** Pupper to it: inverse kinematics, sized and timed for Pupper.
3. **Clean it up** with your coding agent: generated motion floats, slides and starts anywhere.
4. **Replay** it with physics, **upload** it to W&B, and **train** a tracking policy, as in the Tricks notebook.

*Runtime → Change runtime type → **G4** (RTX PRO 6000), then run the cells top to
bottom.* UniMate needs the big GPU: on smaller ones each batch of samples takes
much longer or runs out of memory.

UniMate's code is MIT licensed; its released model weights are
[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/) (non-commercial
use, which a class is).

## 1. Setup

In [ ]:
# Engineering branch: clone once, record its exact revision, and use repo code.
REPO_URL = "https://github.com/ShrishPremkrishna/mts_123.git"
BRANCH = "pupper-foundation"
PINNED_COMMIT = ""  # optionally paste the reviewed 40-character commit SHA

import argparse
import os
import subprocess
import sys
from pathlib import Path

checkout = Path("/content/cs123-foundation")
if not checkout.exists():
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "--filter=blob:none", "--sparse", REPO_URL, str(checkout)], check=True, env=env)
    subprocess.run(["git", "-C", str(checkout), "sparse-checkout", "set", "pupper-mjlab"], check=True, env=env)
origin = subprocess.check_output(["git", "-C", str(checkout), "remote", "get-url", "origin"], text=True).strip()
assert origin == REPO_URL, f"Existing checkout belongs to {origin}; use a fresh runtime."
if PINNED_COMMIT:
    subprocess.run(["git", "-C", str(checkout), "fetch", "origin", PINNED_COMMIT], check=True)
    subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", PINNED_COMMIT], check=True)
else:
    branch = subprocess.check_output(["git", "-C", str(checkout), "branch", "--show-current"], text=True).strip()
    assert branch == BRANCH, f"Existing checkout is {branch}; pin an exact commit or use a fresh runtime."
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print("Training source revision:", revision)
os.chdir(checkout / "pupper-mjlab")
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(checkout / "pupper-mjlab" / "src"))
assert not any(k == "mjlab" or k.startswith("mjlab.") for k in sys.modules), "Restart runtime before changing source revisions."

# Set PUPPER_ARTIFACT_ROOT to an already-mounted persistent directory when available.
# The default /content directory is temporary; W&B also receives checkpoints/policies.
ARTIFACT_ROOT = Path(os.environ.get("PUPPER_ARTIFACT_ROOT", "/content/foundation-artifacts"))
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print("Artifacts:", ARTIFACT_ROOT, "(temporary unless configured to mounted storage)")

def run_foundation(profile, motion=None, iterations=None, num_envs=None):
    from mjlab.scripts.pupper_foundation import train
    return train(argparse.Namespace(profile=profile, output=ARTIFACT_ROOT, motion=motion,
        iterations=iterations, num_envs=num_envs, seed=42, logger="wandb", project="mjlab",
        resume_run=None, resume_checkpoint=None))


## 2. Install UniMate

UniMate gets its own Python environment (it pins its own PyTorch), and the
model weights download from Hugging Face: about 3 minutes. It runs as a
separate process that this notebook talks to.

In [ ]:
UNIMATE_DIR = "/content/UniMate"
!bash src/mjlab/tasks/tracking/config/pupper/unimate/install.sh {UNIMATE_DIR}

In [ ]:
from pathlib import Path

import mediapy
import numpy as np

from mjlab.tasks.tracking.config.pupper import motion_design as md
from mjlab.tasks.tracking.config.pupper import unimate as U

if "um" in globals():
  um.close()  # rerunning this cell: stop the old worker first
um = U.UniMate(UNIMATE_DIR)  # loads the model onto the GPU
print("UniMate ready")

## 3. Describe your move ✏️

Each prompt is **2 seconds** of motion. List several and UniMate chains them:
each one starts where the one before ends.

Prompts that work start with **"An object"** and describe one motion, not the
character: *"An object sits down."*, not *"A happy puppy sits down on the
grass."* (UniMate learned from captions written that way, for every kind of
animal and object.) Some starting points:

- *An object sits down.* / *An object lies down.* / *An object stands up.*
- *An object bows down with its chest low and its rear up.*
- *An object lowers its body and extends a front paw.*
- *An object shakes its body.* / *An object turns around to the left.*
- *An object walks forward.* / *An object wags its tail.* (Pupper has no tail. What happens?)

Samples differ a lot, so generate several and pick. `CFG` is how strongly the
prompt is followed: 3 is the default; try 5-7 if your prompt is ignored, lower
for calmer motion. Change `SEED` for a new batch.

In [ ]:
PROMPTS = [
  "An object sits down.",
]
N_SAMPLES = 4
SEED = 0
CFG = 3.0

raw = um.generate(PROMPTS, n=N_SAMPLES, seed=SEED, cfg=CFG)
print(f"UniMate: {N_SAMPLES} samples of {raw.shape[1] / U.GEN_FPS:.1f} s in {um.seconds:.0f} s")

# Fit Pupper to every sample with the default settings, and show them side by side.
fits = [U.fit_pupper(r) for r in raw]
mediapy.show_video(U.render(fits, [f"sample {i}" for i in range(len(fits))]), fps=U.FPS)
for i, f in enumerate(fits):
  s = f.stats
  print(
    f"sample {i}: max joint speed {s['max_joint_speed_rad_s']:5.2f} rad/s, "
    f"travel {s['travel_m']:.2f} m, IK foot error up to {s['ik_foot_error_worst_cm']:.1f} cm, "
    f"{s['frames_with_no_foot_down']} frames with no foot down"
  )
np.savez_compressed(ARTIFACT_ROOT / "unimate_samples.npz", positions=raw)
from mjlab.scripts.pupper_foundation import write_json, source_manifest
write_json(ARTIFACT_ROOT / "unimate_generation.json", {"prompts": PROMPTS, "samples": N_SAMPLES,
    "seed": SEED, "cfg": CFG, "source": source_manifest(), "state": "generated_uninspected"})


The lines are UniMate's skeleton (left side red, right side blue, spine
orange); grey is Pupper fitted to it. Where they disagree, Pupper couldn't
reach: its legs are shorter and stiffer than any animal's.

## 4. Fit it to Pupper

Pick a sample and tune how it fits:

- `STRIDE`: UniMate's animals take big steps. This scales steps and travel
  (planted feet stay planted). Pupper's tricks travel little: 0.6 or less.
- `DURATIONS`: seconds for each prompt (2 s each by default); `SLOW` stretches
  everything. Slower is easier to track and gentler on the motors.
- `TRIM_START` / `TRIM_END`: seconds cut off either end. `HOLD`: seconds to
  keep the final pose (to stay seated, say).

In [ ]:
SAMPLE = 0
STRIDE = 0.6
DURATIONS = None  # e.g. [2.0, 3.0]: one per prompt
SLOW = 1.0
TRIM_START, TRIM_END = 0.0, 0.0
HOLD = 0.0

fit = U.fit_pupper(
  raw[SAMPLE], durations=DURATIONS, stride=STRIDE, slow=SLOW,
  trim_start=TRIM_START, trim_end=TRIM_END, hold=HOLD,
)
mediapy.show_video(U.render([fit], [f"sample {SAMPLE}"], width=640, height=480), fps=U.FPS)
print(fit)

Read the report the same way as for a designed motion: nothing below the ground
(`lowest_*` not negative), no joint at its limit, joint speeds well under
15 rad/s. Then the generated-motion checks: a large IK error means Pupper can't
make that pose, and `frames_with_no_foot_down` counts frames where Pupper
floats in the air without jumping.

## 5. Clean it up (with your coding agent) 🧹

Generated motion is a first draft. Typical problems:

- It **starts and ends anywhere**. Tricks start standing (or lying down) and
  end in a pose the next move can start from.
- It **floats**: whole stretches with no foot on the ground.
- It **slides**: a foot that should be planted drifts.
- It **rears up** on the hind legs, or **lifts both front paws**: hard to
  balance for a 2 kg robot.

Fix what matters for your move in `clean_up`, a function from one
`md.Motion` to another (root position, root orientation and 12 joint angles
at 50 Hz). This is a good job for a coding agent: give it the report above,
`motion_design.py` (its keyframe tools and `md.stand(...)` / `md.build_motion`
help with blends from a standing pose), and say what you want. Then check the
report and the video again.

In [ ]:
from mjlab.tasks.tracking.config.pupper.unimate.repair import repair_motion
from mjlab.scripts.pupper_foundation import write_json
fit.motion.save_npz(ARTIFACT_ROOT / "selected_before_repair.npz")
motion = repair_motion(fit.motion)
for key, value in motion.report.items():
    print(f"{key:36} {value}")
write_json(ARTIFACT_ROOT / "selected_repair_report.json", {"sample": SAMPLE, "before": fit.stats,
    "after": motion.report, "physics_validated": False, "approved_for_hardware": False})
assert motion.report["geometry_pass"], "Inspect and revise this candidate before training: " + str(motion.report["geometry_issues"])


## 6. Replay it with physics

Same check as in the Tricks notebook: the robot's trick-player gains, no
policy. **HOLDS UP** means the recording works as is; **WOBBLES** or
**FALLS** means it needs a trained policy (or a gentler motion).

In [ ]:
TRICK_NAME = "my_unimate_trick"  # lowercase, no spaces: the name on W&B and on the robot

OUT = Path("/content/tricks")
OUT.mkdir(exist_ok=True)
MOTION_FILE = motion.save_npz(OUT / f"{TRICK_NAME}.npz")
motion.save_robot_csv(OUT / f"{TRICK_NAME}_robot.csv")
print("saved", MOTION_FILE)
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_FILE}

To watch it (public link; stop the cell when you're done):

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_FILE} --viewer viser --share True --loops 3

## 7. Upload to W&B and train 🚀

From here on it is the Tricks notebook's loop: log in with the W&B account your
Pupper uses, upload the motion, and train a tracking policy on it (about 15
minutes). Then score it with `play_motion --checkpoint` and fetch it on the
robot with `get_trick.py`: sections 7 and 8 of `CS123_Pupper_Tricks.ipynb`.

Done generating? Free the GPU for training first: the next cell stops UniMate.

In [ ]:
um.close()  # free the generation model's GPU memory before RL
import wandb
wandb.login()
!python -m mjlab.tasks.tracking.config.pupper.upload_motion {MOTION_FILE} --name {TRICK_NAME}


In [ ]:
RUN_DIR = run_foundation("generated_trick", motion=MOTION_FILE)
CHECKPOINT = max(RUN_DIR.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
subprocess.run([sys.executable, "-m", "mjlab.scripts.pupper_foundation", "score",
    "--motion", str(RUN_DIR / MOTION_FILE.name), "--checkpoint", str(CHECKPOINT),
    "--output", str(RUN_DIR / "evaluation"), "--suite"], check=True)


Keep what you made: **Files** (left bar) → `/content/tricks` → download, or
copy it to Google Drive. Colab deletes `/content` when the runtime
disconnects. When you're done: **Runtime → Disconnect**.